> **ARCHIVED research notebook** - kept for provenance and understanding only.
> It is **superseded by the `cloaq` package** (`src/cloaq/`) and may contain known bugs; see [`research/legacy-notebooks/README.md`](../README.md).
> Saved outputs were cleared and local paths removed. QASMBench paths are relative to your own checkout.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

NUM_TRIALS = 100
SHOTS = 1024
BENCHMARK_DIR = "<path-to-your-qasm-folder>" 
BENCHMARK_FILES = [
    "phase_obfuscation_adder_n4.qasm",
    "phase_obfuscation_basis_change_n3.qasm",
    "phase_obfuscation_basis_trotter_n4.qasm",
    "phase_obfuscation_fredkin_n3.qasm",
    "phase_obfuscation_wstate_n3.qasm"
]


def total_variation_distance_from_counts(counts1, counts2):
    all_states = set(counts1.keys()).union(set(counts2.keys()))
    p = np.zeros(len(all_states))
    q = np.zeros(len(all_states))
    state_to_index = {state: i for i, state in enumerate(all_states)}
    for state, count in counts1.items():
        p[state_to_index[state]] = count
    for state, count in counts2.items():
        q[state_to_index[state]] = count
    tvd = (1 / (2 * SHOTS)) * np.sum(np.abs(p - q))
    return tvd

def obfuscate_circuit(qc: QuantumCircuit) -> QuantumCircuit:
    from qiskit.circuit.library import U3Gate
    from numpy.random import uniform

    num_qubits = qc.num_qubits
    obf_qc = QuantumCircuit(num_qubits)

    for qubit in range(num_qubits):
        theta, phi, lam = uniform(0, 2 * np.pi, 3)
        obf_qc.append(U3Gate(theta, phi, lam), [qubit])

    obf_qc.compose(qc, inplace=True)
    return obf_qc

def run_trial(qc_original: QuantumCircuit) -> float:
    qc_locked = obfuscate_circuit(qc_original)

    qc_original.measure_all()
    qc_locked.measure_all()

    simulator = AerSimulator()
    tqc_original = transpile(qc_original, simulator)
    tqc_locked = transpile(qc_locked, simulator)

    result_original = simulator.run(tqc_original, shots=SHOTS).result()
    result_locked = simulator.run(tqc_locked, shots=SHOTS).result()

    counts_original = result_original.get_counts()
    counts_locked = result_locked.get_counts()

    return total_variation_distance_from_counts(counts_original, counts_locked)

def run_benchmark_on_file(file_path: str) -> list:
    print(f"Running benchmark: {file_path}")
    qc = QuantumCircuit.from_qasm_file(file_path)
    return [run_trial(qc.copy()) for _ in range(NUM_TRIALS)]


def main():
    all_tvd_values = []

    for filename in benchmark_files:
        file_path = os.path.join(BENCHMARK_DIR, filename)
        print(f"Processing file: {file_path}")   # <-- Added print here
        tvds = run_benchmark_on_file(file_path)
        all_tvd_values.extend(tvds)

    plt.figure(figsize=(10, 6))
    plt.hist(all_tvd_values, bins=30, edgecolor='black')
    plt.title("Histogram of Total Variation Distance (TVD) Across All Benchmarks")
    plt.xlabel("TVD")
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def main(benchmark_files=None):
    if benchmark_files is None:
        benchmark_files = BENCHMARK_FILES  # fallback to default

    all_tvd_values = []

    for filename in benchmark_files:
        file_path = os.path.join(BENCHMARK_DIR, filename)
        print(f"Processing file: {file_path}")
        tvds = run_benchmark_on_file(file_path)
        all_tvd_values.extend(tvds)

    plt.figure(figsize=(10, 6))
    plt.hist(all_tvd_values, bins=30, edgecolor='black')
    plt.title("Histogram of Total Variation Distance (TVD) Across All Benchmarks")
    plt.xlabel("TVD")
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

main()